In [1]:
library(tidyverse)

# =============================================================================
# 1. READ AND COMBINE DATA FROM FOUR LABS
# =============================================================================

hplc_bbrs  <- read.csv("BCO-DMO/hplc_bbrs.csv", na.strings = "nd")
hplc_mote  <- read.csv("BCO-DMO/hplc_mote.csv", na.strings = "nd")
hplc_hpl   <- read.csv("BCO-DMO/hplc_hpl.csv", na.strings = "nd")
hplc_ngsfc <- read.csv("BCO-DMO/hplc_ngsfc.csv", na.strings = "nd")

hplc_bbrs$source  <- "BBRS"
hplc_mote$source  <- "MOTE"
hplc_hpl$source   <- "HPL"
hplc_ngsfc$source <- "NGSFC"

pigments <- c("Pras", "Lut", "Fuco", "Perid", "Allo", "But_fuco",
              "Hex_fuco", "Zea", "Tot_Chl_b", "DP", "Tot_Chl_a",
              "TChl", "Chl_c1c2", "Chl_c3")

# The 7 diagnostic pigments used for size fractions (Uitz et al. 2006; Lorenzoni et al. 2015)
dp_pigments <- c("Fuco", "Perid", "Allo", "But_fuco", "Hex_fuco", "Zea", "Tot_Chl_b")

fulldat <- c(pigments, "Date", "depth", "source")

HPLC_ds <- rbind(hplc_bbrs[fulldat],
                 hplc_mote[fulldat],
                 hplc_hpl[fulldat],
                 hplc_ngsfc[fulldat])

# Parse date and drop original Date column
HPLC_ds$date <- as.Date(as.character(HPLC_ds$Date), format = "%Y%m%d")
HPLC_ds <- HPLC_ds %>% select(-Date)

# Set sentinel values (-9999 etc.) to NA
HPLC_ds[pigments][HPLC_ds[pigments] < 0] <- NA

# =============================================================================
# 2. ZERO-FILL ALLOXANTHIN AND PERIDININ ONLY
#    These two pigments are most frequently below detection limit:
#    - Allo: ~99% NA on dates not available under strict processing
#    - Perid: ~96% NA; very low concentrations when present
#    Both contribute to nano/micro numerators (not pico), and 
#    comparison analysis confirmed minimal impact on size fractions
#    (MAE < 1 pp on overlapping dates, no systematic bias in pico).
#    Other DP pigments (Zea, Tot_Chl_b, Fuco, Hex_fuco, But_fuco)
#    retain NA to avoid artificially suppressing size class estimates.
# =============================================================================

# Only zero-fill pigments with justifiably low concentrations
# and minimal impact on size fractions (see comparison analysis)
safe_to_zerofill <- c("Allo", "Perid")
HPLC_ds[safe_to_zerofill][is.na(HPLC_ds[safe_to_zerofill])] <- 0

# =============================================================================
# 3. BUILD SOURCE LOOKUP (collapsed string per date, before aggregation)
# =============================================================================

source_lookup <- HPLC_ds %>%
  group_by(date) %>%
  summarise(source = paste(sort(unique(source)), collapse = "+"), .groups = "drop")

# =============================================================================
# 4. AGGREGATE TO CLEAN MESH (one row per date/depth, mean of duplicates)
# =============================================================================

Mesh_HPLC <- HPLC_ds %>%
  select(date, depth, all_of(pigments)) %>%
  group_by(date, depth) %>%
  summarise(across(everything(), ~mean(.x, na.rm = TRUE)), .groups = "drop")

# =============================================================================
# 5. INTERPOLATE EACH PIGMENT OVER DEPTH
# =============================================================================

source('interpolateData.r')

hplc_temp_store <- list()

for (variable in pigments) {
  hplc_temp_store[[variable]] <- interpolateData(
    Mesh_HPLC, variable,
    noofNA = 20,
    output_type = 'integrated'
  )
  names(hplc_temp_store[[variable]])[1] <- variable
}

hplc_ds_cleaned <- hplc_temp_store %>%
  reduce(full_join, by = "date")

# =============================================================================
# 6. RE-ATTACH SOURCE INFO (one row per date, no duplication)
# =============================================================================

hplc_ds_cleaned_source <- left_join(hplc_ds_cleaned, source_lookup, by = "date")

# =============================================================================
# 7. MERGE DUPLICATE MONTHLY CRUISES (Feb & Jul 1997)
# =============================================================================

# Identify all months with more than one sampling date
monthly_duplicates <- hplc_ds_cleaned_source %>%
  mutate(date_month = floor_date(date, "month")) %>%
  group_by(date_month) %>%
  filter(n() > 1)

cat("Monthly duplicates found:\n")
print(monthly_duplicates %>% select(date, source))

# Average duplicates within each month
dup_means <- monthly_duplicates %>%
  summarise(
    across(where(is.numeric), ~mean(.x, na.rm = TRUE)),
    source = paste(sort(unique(source)), collapse = "+"),
    date = mean(date),
    .groups = "drop"
  ) %>%
  select(-date_month)

# Remove original duplicate dates and append averaged rows
dup_dates <- monthly_duplicates$date

hplc_ds_nodups <- hplc_ds_cleaned_source %>%
  filter(!date %in% dup_dates) %>%
  bind_rows(dup_means) %>%
  arrange(date)

# =============================================================================
# 8. CALCULATE SIZE FRACTIONS AND ABSOLUTE BIOMASS
#    Following Lorenzoni et al. 2015 / Uitz et al. 2006:
#    DP = 1.41*Fuco + 1.41*Perid + 0.60*Allo + 0.35*But_fuco
#         + 1.27*Hex_fuco + 0.86*Zea + 1.01*Tot_Chl_b
#    micro = (1.41*Fuco + 1.41*Perid) / DP
#    nano  = (0.60*Allo + 0.35*But_fuco + 1.27*Hex_fuco) / DP
#    pico  = (0.86*Zea + 1.01*Tot_Chl_b) / DP
#    Absolute biomass per size class = fraction * Tot_Chl_a
# =============================================================================

hplc_final_1 <- hplc_ds_nodups %>%
  mutate(
    DP2 = 1.41 * Fuco + 1.41 * Perid + 0.60 * Allo +
          0.35 * But_fuco + 1.27 * Hex_fuco + 0.86 * Zea + 1.01 * Tot_Chl_b,
    # Guard against division by zero / near-zero DP
    DP2 = ifelse(DP2 < 0.001, NA, DP2),
    # Relative size fractions
    micro = (1.41 * Fuco + 1.41 * Perid) / DP2,
    nano  = (0.60 * Allo + 0.35 * But_fuco + 1.27 * Hex_fuco) / DP2,
    pico  = (0.86 * Zea + 1.01 * Tot_Chl_b) / DP2,
    # Absolute biomass per size class (integrated Chl a, mg m-2)
    micro_abs = micro * Tot_Chl_a,
    nano_abs  = nano  * Tot_Chl_a,
    pico_abs  = pico  * Tot_Chl_a,
    time_month = format(date, format = "%m-%Y")
  )


# =============================================================================
# 8b. DERIVED SIZE-SPECTRAL METRICS
#     Size centroid: biomass-weighted mean log-size, using relative fractions
#       as weights and geometric-mean ESD per size class.
#       Range: log10(0.63) ≈ -0.20 (all pico) to log10(63) ≈ 1.80 (all micro)
#       Higher values = larger community mean cell size.
#
#     Shannon size diversity: H' applied to the three relative size fractions.
#       Range: 0 (one class = 100%) to ln(3) ≈ 1.10 (perfect evenness).
#       Captures how evenly biomass is distributed across size classes,
#       independent of direction (large vs small).
#
#     Representative ESD per class (geometric mean of bin boundaries):
#       pico:  0.2–2 µm   -> 0.63 µm
#       nano:  2–20 µm    -> 6.3 µm
#       micro: 20–200 µm  -> 63 µm
# =============================================================================

hplc_final <- hplc_final_1 %>%
  mutate(
    # Size centroid (biomass-weighted mean log10 ESD)
    size_centroid = micro * log10(63) + nano * log10(6.3) + pico * log10(0.63),

    # Shannon size diversity (using natural log, max = ln(3) ≈ 1.10)
    size_shannon = -( ifelse(micro > 0, micro * log(micro), 0) +
                      ifelse(nano  > 0, nano  * log(nano),  0) +
                      ifelse(pico  > 0, pico  * log(pico),  0) ),

    # NBSS slope (Normalized Biomass Size Spectrum)
    # Linear regression of log10(biomass) vs log10(ESD) across the three bins.
    # Bin widths in log10 space are equal (each ~1 decade), so normalization
    # by bin width does not affect the slope.
    # Positive slope = large cells carry more biomass (top-heavy spectrum)
    # Negative slope = small cells carry more biomass (bottom-heavy spectrum)
    # With 3 equally-spaced points, OLS slope simplifies to:
    #   (log10(micro_abs) - log10(pico_abs)) / (log10(63) - log10(0.63))
    nbss_slope = ifelse(
      micro_abs > 0 & pico_abs > 0,
      (log10(micro_abs) - log10(pico_abs)) / (log10(63) - log10(0.63)),
      NA_real_
    )
  )


# =============================================================================
# 9. DIAGNOSTICS
# =============================================================================

cat("\n--- Final dataset structure ---\n")
str(hplc_final)

cat("\n--- NA counts in key columns ---\n")
hplc_final %>%
  summarise(across(c(Tot_Chl_a, Fuco, DP2, micro, nano, pico, micro_abs, nano_abs, pico_abs),
                   ~sum(is.na(.x)))) %>%
  print()

cat("\n--- Size fraction sanity check (should sum to ~1) ---\n")
hplc_final %>%
  filter(!is.na(micro)) %>%
  mutate(size_sum = micro + nano + pico) %>%
  summarise(mean = mean(size_sum, na.rm = TRUE),
            min  = min(size_sum, na.rm = TRUE),
            max  = max(size_sum, na.rm = TRUE)) %>%
  print()

cat("\n--- Coverage summary ---\n")
n_total <- nrow(hplc_final)
n_sf    <- sum(!is.na(hplc_final$DP2))
n_abs   <- sum(!is.na(hplc_final$micro_abs))
cat(sprintf("Total dates:                      %d\n", n_total))
cat(sprintf("Dates with size fractions:        %d (%.0f%%)\n", n_sf, 100 * n_sf / n_total))
cat(sprintf("Dates with absolute biomass:      %d (%.0f%%)\n", n_abs, 100 * n_abs / n_total))

# =============================================================================
# 10. SAVE
# =============================================================================

write.csv(hplc_final, "processed/HPLC_interpolated_sizes.csv")
saveRDS(hplc_final, "processed/HPLC_interpolated_sizes.rds")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Lade nötiges Paket: gsw



Monthly duplicates found:


Adding missing grouping variables: `date_month`


# A tibble: 4 × 3
# Groups:   date_month [2]
  date_month date       source
  <date>     <date>     <chr> 
1 1997-02-01 1997-02-14 BBRS  
2 1997-02-01 1997-02-23 BBRS  
3 1997-07-01 1997-07-16 BBRS  
4 1997-07-01 1997-07-08 BBRS  

--- Final dataset structure ---
'data.frame':	169 obs. of  27 variables:
 $ Pras         : num  NA NA NA NA NA ...
 $ date         : Date, format: "1995-12-13" "1996-01-12" ...
 $ Lut          : num  NA NA NA NA NA NA NA NA NA NA ...
 $ Fuco         : num  1.54 25.67 21.22 29.12 NA ...
 $ Perid        : num  0.391 4.249 0.945 1.555 1.515 ...
 $ Allo         : num  0.095 0.187 0.333 0.279 0 ...
 $ But_fuco     : num  NA NA 0.381 0.158 NA ...
 $ Hex_fuco     : num  1.99 1.98 2.21 1.84 NA ...
 $ Zea          : num  0.697 NA NA NA NA ...
 $ Tot_Chl_b    : num  4.97 5.19 3.75 2.48 NA ...
 $ DP           : num  NA NA NA NA NA NA NA NA NA NA ...
 $ Tot_Chl_a    : num  12.7 43.4 27.8 28 NA ...
 $ TChl         : num  NA NA NA NA NA NA NA NA NA NA ...
 $ Chl_c1c2     